# LSSTCam — HEALPix Visit Count Sky Maps

- Creation date : 2026-03-09
- Based on : `2026-03-09_ConsDB_LSSTCam.ipynb`

This notebook queries ConsDB to retrieve LSSTCam visit data (same query as the parent notebook),  
then produces **HEALPix sky maps of visit counts** using `healpy`:

1. All visits combined (Mollweide projection)
2. One map per LSST band (*u, g, r, i, z, y*)

The Galactic plane and Deep Drilling Fields are overlaid on each map.

## 1. Imports

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import healpy as hp

from astropy.table import join
from astropy.coordinates import SkyCoord
import astropy.units as u

from lsst.summit.utils import ConsDbClient

%matplotlib widget

print(f'healpy  version : {hp.__version__}')
print(f'numpy   version : {np.__version__}')

## 2. Configuration

In [ ]:
# ── Instrument ────────────────────────────────────────────────────────────────
instrument = 'LSSTCam'

# ── HEALPix resolution ────────────────────────────────────────────────────────
# NSIDE=64  → pixel size ~ 0.92 deg  (good overview)
# NSIDE=128 → pixel size ~ 0.46 deg  (finer, still fast)
# NSIDE=256 → pixel size ~ 0.23 deg  (matches LSSTCam FOV ~3.5 deg diameter)
NSIDE = 64
NPIX  = hp.nside2npix(NSIDE)
print(f'NSIDE={NSIDE}  →  {NPIX} pixels,  pixel size ~ {np.degrees(hp.nside2resol(NSIDE)):.2f} deg')

# ── Bands ─────────────────────────────────────────────────────────────────────
BANDS = list('ugrizy')
BAND_COLORS = {
    'u': '#9b59b6', 'g': '#2ecc71', 'r': '#e74c3c',
    'i': '#e67e22', 'z': '#3498db', 'y': '#795548',
}

# ── Matplotlib defaults ───────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.figsize' : (12, 6),
    'axes.labelsize' : 'x-large',
    'axes.titlesize' : 'x-large',
    'xtick.labelsize': 'x-large',
    'ytick.labelsize': 'x-large',
})

# ── Deep Drilling Fields ──────────────────────────────────────────────────────
DDF_NAMES  = ['XMM-LSS', 'COSMOS', 'ECDFS', 'ELAIS-S1', 'EDFS_a', 'EDFS_b']
DDF_RA_DEG = np.array([35.57, 150.11, 52.98, 9.45, 58.90, 63.60])
DDF_DEC_DEG= np.array([-4.82,   2.23,-28.12,-44.02,-49.32,-47.60])

print('Configuration done.')

## 3. ConsDB connection & data retrieval

Identical query to the parent notebook `2026-03-09_ConsDB_LSSTCam.ipynb`.

In [ ]:
os.environ['no_proxy'] += ',.consdb'
url    = 'http://consdb-pq.consdb:8080/consdb'
consdb = ConsDbClient(url)
print('ConsDB client created.')

In [ ]:
# Same queries as in the parent notebook
visits    = consdb.query('SELECT * FROM cdb_lsstcam.visit1 WHERE day_obs >= 20250415')
visits_ql = consdb.query('SELECT * FROM cdb_lsstcam.visit1')

merged_visits = join(visits, visits_ql, keys='visit_id', join_type='inner')

df_visits = visits.to_pandas()
print(f'Visits retrieved: {len(df_visits)}')
print(f'Columns: {list(df_visits.columns)}')

## 4. Data cleaning

Identical cleaning steps as in the parent notebook.

In [ ]:
# Remove engineering / pinhole filters
bad_filters = {'other', 'none', 'other:pinhole'}
df_visits = df_visits[~df_visits['physical_filter'].isin(bad_filters)]

# Remove rows with missing pointing coordinates
df_visits = df_visits.dropna(subset=['s_ra', 's_dec']).reset_index(drop=True)

print(f'Visits after cleaning : {len(df_visits)}')
print(f'Bands present         : {sorted(df_visits["band"].dropna().unique())}')
print(f'RA  range (deg)       : [{df_visits["s_ra"].min():.2f}, {df_visits["s_ra"].max():.2f}]')
print(f'Dec range (deg)       : [{df_visits["s_dec"].min():.2f}, {df_visits["s_dec"].max():.2f}]')

## 5. Helper functions

In [ ]:
def visits_to_healpix_map(ra_deg, dec_deg, nside=NSIDE):
    """
    Build a HEALPix visit-count map from pointing coordinates.

    Parameters
    ----------
    ra_deg, dec_deg : array-like, degrees
    nside : int — HEALPix NSIDE parameter

    Returns
    -------
    hpmap : np.ndarray of shape (hp.nside2npix(nside),)
        Visit count per pixel; pixels with no visits are set to hp.UNSEEN.
    """
    ra  = np.asarray(ra_deg,  dtype=float)
    dec = np.asarray(dec_deg, dtype=float)

    # HEALPix uses colatitude θ = 90-dec (in degrees → radians)
    theta = np.radians(90.0 - dec)
    phi   = np.radians(ra)

    ipix = hp.ang2pix(nside, theta, phi)

    hpmap = np.zeros(hp.nside2npix(nside), dtype=float)
    np.add.at(hpmap, ipix, 1)

    # Mask empty pixels
    hpmap[hpmap == 0] = hp.UNSEEN
    return hpmap


def galactic_plane_coords():
    """Return (ra_rad, dec_rad) of the Galactic plane in ICRS for Mollweide overlay."""
    gal_l   = np.linspace(-180., 180., 720)
    gal_b   = np.zeros(720)
    gp      = SkyCoord(l=gal_l * u.deg, b=gal_b * u.deg, frame='galactic').icrs
    ra_rad  = np.radians(gp.ra.deg)
    dec_rad = np.radians(gp.dec.deg)
    # Mollweide convention: RA in (-π, π], increasing to the left
    ra_rad  = np.remainder(ra_rad + 2*np.pi, 2*np.pi)
    ra_rad[ra_rad > np.pi] -= 2*np.pi
    ra_rad  = -ra_rad
    return ra_rad, dec_rad


def ddf_mollweide_coords(ra_deg, dec_deg):
    """Convert DDF (RA, Dec) in degrees to Mollweide (ra_rad, dec_rad)."""
    ra_rad  = np.radians(ra_deg)
    dec_rad = np.radians(dec_deg)
    ra_rad  = np.remainder(ra_rad + 2*np.pi, 2*np.pi)
    ra_rad[ra_rad > np.pi] -= 2*np.pi
    ra_rad  = -ra_rad
    return ra_rad, dec_rad


# Pre-compute overlays
GP_RA_RAD, GP_DEC_RAD         = galactic_plane_coords()
DDF_RA_RAD, DDF_DEC_RAD       = ddf_mollweide_coords(DDF_RA_DEG, DDF_DEC_DEG)

print('Helper functions defined.')

In [ ]:
def plot_healpix_map(hpmap, title, cmap='YlOrRd', unit='N visits',
                     nside=NSIDE, show_gp=True, show_ddf=True):
    """
    Display a HEALPix visit-count map in Mollweide projection using healpy.mollview,
    then overlay the Galactic plane and Deep Drilling Fields via matplotlib.

    Parameters
    ----------
    hpmap    : np.ndarray — healpy map (visit counts, UNSEEN for empty pixels)
    title    : str        — figure title
    cmap     : str        — matplotlib colormap name
    unit     : str        — colorbar label
    show_gp  : bool       — overlay Galactic plane
    show_ddf : bool       — overlay Deep Drilling Fields
    """
    # healpy.mollview renders into the current figure
    hp.mollview(
        hpmap,
        title=title,
        cmap=cmap,
        unit=unit,
        norm=None,
        bgcolor='white',
        badcolor='lightgrey',
        flip='astro',          # RA increases to the left (standard convention)
        coord='C',             # Celestial (ICRS)
    )
    hp.graticule(dpar=30, dmer=30, alpha=0.4, lw=0.5)

    # Overlay on the healpy axes
    ax = plt.gca()

    if show_gp:
        # Sort by RA to avoid wrap-around lines
        idx = np.argsort(GP_RA_RAD)
        ax.plot(GP_RA_RAD[idx], GP_DEC_RAD[idx],
                '.', ms=1, color='steelblue', label='Galactic Plane', zorder=5)

    if show_ddf:
        for i, name in enumerate(DDF_NAMES):
            ax.plot(DDF_RA_RAD[i], DDF_DEC_RAD[i],
                    '*', ms=10, color='gold', markeredgecolor='black',
                    zorder=10)
            ax.text(DDF_RA_RAD[i] + 0.04, DDF_DEC_RAD[i] + 0.04,
                    name, fontsize=7, color='gold',
                    ha='left', va='bottom', fontweight='bold', zorder=11)

    if show_gp or show_ddf:
        ax.legend(loc='lower right', fontsize=8, markerscale=2)

    plt.tight_layout()
    plt.show()


print('Plotting function defined.')

## 6. HEALPix map — all visits

In [ ]:
hpmap_all = visits_to_healpix_map(
    df_visits['s_ra'].values,
    df_visits['s_dec'].values,
    nside=NSIDE
)

n_pix_obs = int((hpmap_all != hp.UNSEEN).sum())
n_vis_tot = int(df_visits.shape[0])
print(f'Total visits          : {n_vis_tot}')
print(f'Observed pixels       : {n_pix_obs} / {NPIX}')
print(f'Max visits per pixel  : {int(hpmap_all[hpmap_all != hp.UNSEEN].max())}')

plot_healpix_map(
    hpmap_all,
    title=f'{instrument} — All bands — Visit count map (NSIDE={NSIDE})',
    cmap='YlOrRd',
    unit='N visits',
)

## 7. HEALPix maps — one per band

Each sub-plot uses the same `NSIDE` and the same Mollweide projection.

In [ ]:
# Build one map per band
hpmaps_per_band = {}

for band in BANDS:
    df_b = df_visits[df_visits['band'] == band]
    if df_b.empty:
        print(f'  Band {band}: no data, skipping.')
        continue
    hpmaps_per_band[band] = visits_to_healpix_map(
        df_b['s_ra'].values,
        df_b['s_dec'].values,
        nside=NSIDE
    )
    n_pix = int((hpmaps_per_band[band] != hp.UNSEEN).sum())
    n_vis = int(df_b.shape[0])
    vmax  = int(hpmaps_per_band[band][hpmaps_per_band[band] != hp.UNSEEN].max())
    print(f'  Band {band}: {n_vis:5d} visits | {n_pix:4d} pixels | max {vmax} visits/pixel')

In [ ]:
# Display one Mollweide map per available band
for band, hpmap_b in hpmaps_per_band.items():
    plot_healpix_map(
        hpmap_b,
        title=f'{instrument} — Band {band} — Visit count map (NSIDE={NSIDE})',
        cmap='YlOrRd',
        unit=f'N visits — band {band}',
    )

## 8. Summary panel — all 6 bands side by side

In [ ]:
bands_available = [b for b in BANDS if b in hpmaps_per_band]
n_bands = len(bands_available)

if n_bands == 0:
    print('No band data available.')
else:
    # healpy.mollview draws into plt.figure(), so we use a 2-column grid
    ncols = 2
    nrows = (n_bands + 1) // ncols

    for idx, band in enumerate(bands_available):
        plt.figure(figsize=(10, 5))
        hp.mollview(
            hpmaps_per_band[band],
            title=f'Band {band}',
            cmap='YlOrRd',
            unit='N visits',
            bgcolor='white',
            badcolor='lightgrey',
            flip='astro',
            coord='C',
        )
        hp.graticule(dpar=30, dmer=30, alpha=0.3, lw=0.5)

        ax = plt.gca()
        # Galactic plane
        idx_sorted = np.argsort(GP_RA_RAD)
        ax.plot(GP_RA_RAD[idx_sorted], GP_DEC_RAD[idx_sorted],
                '.', ms=1, color='steelblue', zorder=5)
        # DDFs
        for i, name in enumerate(DDF_NAMES):
            ax.plot(DDF_RA_RAD[i], DDF_DEC_RAD[i],
                    '*', ms=9, color='gold', markeredgecolor='black', zorder=10)
            ax.text(DDF_RA_RAD[i]+0.04, DDF_DEC_RAD[i]+0.04,
                    name, fontsize=6, color='gold',
                    ha='left', va='bottom', fontweight='bold', zorder=11)

        plt.suptitle(f'{instrument} — Band {band}', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()

## 9. Visit count statistics per band

In [ ]:
import pandas as pd

rows = []
# All bands
valid = hpmap_all[hpmap_all != hp.UNSEEN]
rows.append({
    'band'         : 'ALL',
    'n_visits'     : int(df_visits.shape[0]),
    'n_pixels_obs' : len(valid),
    'sky_area_deg2': len(valid) * hp.nside2pixarea(NSIDE, degrees=True),
    'max_visits'   : int(valid.max()),
    'mean_visits'  : float(valid.mean()),
    'median_visits': float(np.median(valid)),
})

# Per band
for band in BANDS:
    df_b = df_visits[df_visits['band'] == band]
    if df_b.empty or band not in hpmaps_per_band:
        continue
    valid_b = hpmaps_per_band[band][hpmaps_per_band[band] != hp.UNSEEN]
    rows.append({
        'band'         : band,
        'n_visits'     : int(df_b.shape[0]),
        'n_pixels_obs' : len(valid_b),
        'sky_area_deg2': len(valid_b) * hp.nside2pixarea(NSIDE, degrees=True),
        'max_visits'   : int(valid_b.max()),
        'mean_visits'  : float(valid_b.mean()),
        'median_visits': float(np.median(valid_b)),
    })

df_stats = pd.DataFrame(rows).set_index('band')
print(f'HEALPix pixel area (NSIDE={NSIDE}): {hp.nside2pixarea(NSIDE, degrees=True):.4f} deg²')
display(df_stats.round(2))

In [ ]:
# Bar chart: total visits per band
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_bands = df_stats.drop('ALL')

# Total visits
ax = axes[0]
colors = [BAND_COLORS.get(b, 'grey') for b in df_bands.index]
ax.bar(df_bands.index, df_bands['n_visits'], color=colors, edgecolor='k', linewidth=0.5)
ax.set_xlabel('Band')
ax.set_ylabel('Total visits')
ax.set_title('Total visits per band')
ax.grid(axis='y', alpha=0.4)

# Sky area covered
ax = axes[1]
ax.bar(df_bands.index, df_bands['sky_area_deg2'], color=colors, edgecolor='k', linewidth=0.5)
ax.set_xlabel('Band')
ax.set_ylabel('Sky area (deg²)')
ax.set_title(f'Sky area covered per band  (NSIDE={NSIDE})')
ax.grid(axis='y', alpha=0.4)

plt.suptitle(instrument, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()